In [ ]:
from pathlib import Path
import re
import pandas as pd

# Change the name to match your CSV file
input_csv = Path("../../data/processed/merged_plays.csv")
output_csv = Path("../../data/processed/penalty_plays.csv")

df = pd.read_csv(input_csv)
df.drop_duplicates(subset="playId", inplace=True)

def extract_penalty(row):
    text = str(row["text"]).lower()

    # Exclude texts that only announce the start or end of the penalty shootout
    is_penalty = (
        "converts the penalty" in text
        or "penalty saved" in text
        or "penalty missed" in text
    )

    if not is_penalty:
        return pd.Series({
            "penalty_position": pd.NA,
            "penalty_direction": pd.NA,
            "save": pd.NA,
            "goal": pd.NA,
            "miss": pd.NA
        })

    # 1 if the goalkeeper saves the penalty
    save = int("penalty saved" in text)

    # Outcome of the penalty kick
    goal = int("converts the penalty" in text)
    miss = int("penalty missed" in text)

    # Search for the direction in the text
    if re.search(r"misses the top left corner|high and wide to the left", text):
        position, direction = 10, "misses top left"
    elif re.search(r"hits the bar|too high", text):
        position, direction = 11, "too high"
    elif re.search(r"misses the top right corner|high and wide to the right", text):
        position, direction = 12, "misses top right"

    elif re.search(r"top right", text):
        position, direction = 1, "top right"
    elif re.search(r"(top|high) centre", text):
        position, direction = 2, "top centre"
    elif re.search(r"top left", text):
        position, direction = 3, "top left"

    elif re.search(r"bottom right", text):
        position, direction = 4, "bottom right"
    elif re.search(r"centre of the goal", text):
        position, direction = 5, "bottom centre"
    elif re.search(r"bottom left", text):
        position, direction = 6, "bottom left"

    elif re.search(r"hits the left post|misses to the left", text):
        position, direction = 13, "misses left"
    elif re.search(r"hits the right post|misses to the right", text):
        position, direction = 14, "misses right"

    # When the text only says "to the right/left", the height is unknown.
    # We classify it as middle height.
    else:
        position, direction = 15, text

    return pd.Series({
        "penalty_position": position,
        "penalty_direction": direction,
        "save": save,
        "goal": goal,
        "miss": miss
    })


# Extract the new variables
new_columns = df.apply(extract_penalty, axis=1)
df = pd.concat([df, new_columns], axis=1)

# Keep only penalty kicks
penalties = df[df["penalty_position"].notna()].copy()

# Use nullable integers to allow missing values
for column in ["penalty_position", "save", "goal", "miss"]:
    penalties[column] = penalties[column].astype("Int64")

# Create the directory and save the result
penalties.to_csv(output_csv, index=False)

print(penalties[
    [
        "eventId",
        "playOrder",
        "text",
        "penalty_position",
        "penalty_direction",
        "save",
        "goal",
        "miss"
    ]
])

print("Saved to:", output_csv.resolve())

In [ ]:
from pathlib import Path
import re
import pandas as pd

# Change the name to match your CSV file
input_csv = Path("../../data/processed_others/merged_plays.csv")
output_csv_others = Path("../../data/processed_others/penalty_plays.csv")

df = pd.read_csv(input_csv)
df.drop_duplicates(subset="playId", inplace=True)


def extract_penalty(row):
    text = str(row["text"]).lower()

    # Exclude texts that only announce the start or end of the penalty shootout
    is_penalty = (
        "converts the penalty" in text
        or "penalty saved" in text
        or "penalty missed" in text
    )

    if not is_penalty:
        return pd.Series({
            "penalty_position": pd.NA,
            "penalty_direction": pd.NA,
            "save": pd.NA,
            "goal": pd.NA,
            "miss": pd.NA
        })


    # 1 if the goalkeeper saves the penalty
    save = int("penalty saved" in text)

    # Outcome of the penalty kick
    goal = int("converts the penalty" in text)
    miss = int("penalty missed" in text)

    # Search for the direction in the text
    if re.search(r"misses the top left corner|high and wide to the left", text):
        position, direction = 10, "misses top left"
    elif re.search(r"hits the bar|too high", text):
        position, direction = 11, "too high"
    elif re.search(r"misses the top right corner|high and wide to the right", text):
        position, direction = 12, "misses top right"

    elif re.search(r"top right", text):
        position, direction = 1, "top right"
    elif re.search(r"(top|high) centre", text):
        position, direction = 2, "top centre"
    elif re.search(r"top left", text):
        position, direction = 3, "top left"

    elif re.search(r"bottom right", text):
        position, direction = 4, "bottom right"
    elif re.search(r"centre of the goal", text):
        position, direction = 5, "bottom centre"
    elif re.search(r"bottom left", text):
        position, direction = 6, "bottom left"

    elif re.search(r"hits the left post|misses to the left", text):
        position, direction = 13, "misses left"
    elif re.search(r"hits the right post|misses to the right", text):
        position, direction = 14, "misses right"

    # When the text only says "to the right/left", the height is unknown.
    # We classify it as middle height.
    else:
        position, direction = 15, text

    return pd.Series({
        "penalty_position": position,
        "penalty_direction": direction,
        "save": save,
        "goal": goal,
        "miss": miss
    })


# Extract the new variables
new_columns = df.apply(extract_penalty, axis=1)
df = pd.concat([df, new_columns], axis=1)

# Keep only penalty kicks
penalties = df[df["penalty_position"].notna()].copy()

# Use nullable integers to allow missing values
for column in ["penalty_position", "save", "goal", "miss"]:
    penalties[column] = penalties[column].astype("Int64")

# Create the directory and save the result
penalties.to_csv(output_csv_others, index=False)

print(penalties[
    [
        "eventId",
        "playOrder",
        "text",
        "penalty_position",
        "penalty_direction",
        "save",
        "goal",
        "miss"
    ]
])

print("Saved to:", output_csv_others.resolve())

In [ ]:
penalties = pd.read_csv(output_csv)
penalties_saved = penalties["save"].sum()
penalties_goal = penalties["goal"].sum()
penalties_missed = penalties["miss"].sum()

print(f"saved: {penalties_saved}, saved in %: {penalties_saved/len(penalties)*100:.2f}%")
print(f"goal: {penalties_goal}, goal in %: {penalties_goal/len(penalties)*100:.2f}%")
print(f"missed: {penalties_missed}, missed in %: {penalties_missed/len(penalties)*100:.2f}%")
print(f"total: {len(penalties)}")

In [ ]:
output_csv_others = Path("../../data/processed_others/penalties.csv")

penalties_others = pd.read_csv(output_csv_others)
penalties_others_saved = penalties_others["save"].sum()
penalties_others_goal = penalties_others["goal"].sum()
penalties_others_missed = penalties_others["miss"].sum()

print(f"saved: {penalties_others_saved}, saved in %: {penalties_others_saved/len(penalties_others)*100:.2f}%")
print(f"goal: {penalties_others_goal}, goal in %: {penalties_others_goal/len(penalties_others)*100:.2f}%")
print(f"missed: {penalties_others_missed}, missed in %: {penalties_others_missed/len(penalties_others)*100:.2f}%")
print(f"total: {len(penalties_others)}")